In [1]:
import os
import numpy as np
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon
from rasterio.features import rasterize
from PIL import Image
import rasterio

In [2]:
# -------------------------
# CONFIG (PROJECTED CRS ONLY)
# -------------------------
TARGET_CRS = "EPSG:3857"
input_geojson = "output/vect/poly/water.geojson"
output_geojson = "output/vect/poly/water.geojson"
output_mask = "output/vect/poly/water.png"

In [3]:
reference_raster = "data/el_harrach_georef.tif"

In [4]:
BUFFER_DIST = 20        # meters
SIMPLIFY_TOL = 0.5     # meters

In [5]:
os.makedirs(os.path.dirname(output_geojson), exist_ok=True)

In [6]:
# -------------------------
# LOAD VECTOR
# -------------------------
gdf = gpd.read_file(input_geojson)

In [7]:
if gdf.empty:
    raise ValueError("❌ GeoJSON is empty")

In [8]:
print("\n--- VECTOR INFO ---")
print("CRS:", gdf.crs)
print("Bounds:", gdf.total_bounds)


--- VECTOR INFO ---
CRS: EPSG:3857
Bounds: [ 346565.48624499 4397861.15299455  349151.80475673 4401855.58488673]


In [9]:
# -------------------------
# LOAD RASTER
# -------------------------
with rasterio.open(reference_raster) as src:
    transform = src.transform
    h, w = src.height, src.width
    raster_crs = src.crs
    raster_bounds = src.bounds

In [10]:
print("\n--- RASTER INFO ---")
print("CRS:", raster_crs)
print("Bounds:", raster_bounds)
print("Shape:", (h, w))


--- RASTER INFO ---
CRS: EPSG:3857
Bounds: BoundingBox(left=346565.48624498915, bottom=4396199.244793627, right=352374.7003946625, top=4401855.584886729)
Shape: (9472, 9728)


In [11]:
# -------------------------
# FORCE CRS ALIGNMENT
# -------------------------
if gdf.crs is None:
    raise ValueError("❌ Input GeoJSON has no CRS")

In [12]:
if str(raster_crs) != TARGET_CRS:
    raise ValueError(f"❌ Expected raster CRS {TARGET_CRS}, got {raster_crs}")

In [13]:
if str(gdf.crs) != TARGET_CRS:
    raise ValueError(f"❌ Expected vector CRS {TARGET_CRS}, got {gdf.crs}")

In [14]:
print("\n--- AFTER REPROJECTION ---")
print("CRS:", gdf.crs)
print("Bounds:", gdf.total_bounds)


--- AFTER REPROJECTION ---
CRS: EPSG:3857
Bounds: [ 346565.48624499 4397861.15299455  349151.80475673 4401855.58488673]


In [15]:
# -------------------------
# OVERLAP CHECK
# -------------------------
vxmin, vymin, vxmax, vymax = gdf.total_bounds
rxmin, rymin, rxmax, rymax = raster_bounds

In [16]:
overlap = not (
    vxmax < rxmin or vxmin > rxmax or
    vymax < rymin or vymin > rymax
)

In [17]:
print("\n--- OVERLAP CHECK ---")
print("Overlap:", overlap)


--- OVERLAP CHECK ---
Overlap: True


In [18]:
if not overlap:
    raise ValueError("❌ Vector and raster do NOT overlap")

In [19]:
# -------------------------
# MERGE + FILL GAPS
# -------------------------
merged = unary_union(gdf.geometry)

In [20]:
filled = merged.buffer(BUFFER_DIST).buffer(-BUFFER_DIST)

In [21]:
if filled.is_empty:
    raise ValueError("❌ Geometry became empty after buffering")

In [22]:
# -------------------------
# REMOVE HOLES
# -------------------------
def remove_holes(geom):
    if isinstance(geom, Polygon):
        return Polygon(geom.exterior)
    elif isinstance(geom, MultiPolygon):
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])
    return geom

In [23]:
filled = remove_holes(filled)

In [24]:
if filled.is_empty:
    raise ValueError("❌ Geometry empty after hole removal")

In [25]:
# -------------------------
# SIMPLIFY
# -------------------------
filled = filled.simplify(SIMPLIFY_TOL)

In [26]:
if filled.is_empty:
    raise ValueError("❌ Geometry empty after simplify")

In [27]:
# -------------------------
# SAVE GEOJSON
# -------------------------
out_gdf = gpd.GeoDataFrame(geometry=[filled], crs=gdf.crs)
out_gdf["class"] = "water"

In [28]:
out_gdf.to_file(output_geojson, driver="GeoJSON")

In [29]:
print(f"\n✅ Saved GeoJSON: {output_geojson}")


✅ Saved GeoJSON: output/vect/poly/water.geojson


In [30]:
# -------------------------
# RASTERIZE
# -------------------------
print("\n--- RASTERIZING ---")


--- RASTERIZING ---


In [31]:
mask = rasterize(
    [(filled, 1)],
    out_shape=(h, w),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

In [32]:
print("Mask sum (should be > 0):", int(mask.sum()))

Mask sum (should be > 0): 1178645


In [33]:
if mask.sum() == 0:
    raise ValueError("❌ Empty mask → geometry not aligned with raster")

In [34]:
# -------------------------
# EXPORT MASK (RED)
# -------------------------
out = np.zeros((h, w, 3), dtype=np.uint8)
out[mask == 1] = [255, 0, 0]

In [35]:
Image.fromarray(out).save(output_mask)

In [36]:
print(f"✅ Saved mask: {output_mask}")
print("\n🎯 Done successfully")

✅ Saved mask: output/vect/poly/water.png

🎯 Done successfully
